In [199]:
import pandas as pd
import numpy as np
import requests

import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression


[тут будет вводная какая-то]

колонка "Строительная готовность" не обработана, там пропуски у новостроек а она оч важная, также

колонка "метро" не обработана (не уверен как лучше заполнять пропуски, коих оч много, и колонка мегаважная)

и колонка "время до метро" не обработана



Разместим наш dataframe на гугл диске, для простоты открытия, и посмотрим на него

In [200]:
url = ('https://drive.google.com/file/d/1P_vseOPvDeJnK2t424d8Wi7uQSbxkYic/view?usp=sharing')
url='https://drive.google.com/uc?id=' + url.split('/')[-2]
df = pd.read_csv(url, sep=';')

/tmp/ipykernel_4322/778542303.py:3: DtypeWarning: Columns (19,45,58,60,64) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url, sep=';')


In [201]:
print(df.head(5))

                                               Адрес  Площадь    Жилая  \
0                 Москва, 2-й Иртышский проезд, к1/4  59,6 м²  21,8 м²   
1  Москва, пос. Коммунарка, улица Александры Мона...  72,2 м²  34,7 м²   
2                      Москва, Кварцевая улица, 2 к4  59,4 м2      NaN   
3  Москва, Матвеевское м-н, Староволынская улица,...    65 м2    30 м2   
4  Москва, пос. Коммунарка, улица Липовый парк, 4 к3  71,6 м2  34,9 м2   

     Кухня Класс жилья   Отделка  Комнат Вид из окна  Санузел совмещён  \
0  20,5 м²      Бизнес  Чистовая     2.0       Улица               2.0   
1  15,7 м²     Комфорт  Чистовая     3.0        Двор               2.0   
2  19,4 м2         NaN       NaN     2.0         NaN               NaN   
3    10 м2         NaN       NaN     2.0         NaN               NaN   
4  17,2 м2         NaN       NaN     2.0         NaN               NaN   

   Подъезд  ... Детская площадка Европланировка Теплоснабжение  \
0      5.0  ...              NaN            

In [202]:
df.shape

(10481, 67)

Видим, что у нас 67 признаков
Займёмся удалением лишних столбцов, которые не несут для нас ценной информации.

Важно, что нам следует оставить признаки, которые могут быть полезны именно для определения цены на квартиру

Таким образом, "Мусором" будет:
Адрес (по нему у нас есть информация по инфраструктуе, в т.ч. метро, поэтому он лишний и не несёт в себе информации)
Подъезд (номер подъезда)
Описание (важная информация из описания уже вынесена в отдельные признаки)

Балкон, балконы, лоджия и лоджии (информация внесена в признак "Количетсво балконов")

ВсегоКвартирвпродаже

Финансируется ли Сбербанком

Материал стен (дублирует колонку материалстен)

Количество этажей (колонка дублируется)

Количество подъездов

Квартира (номер квартиры)

Лифт (в 100% объектах значение равно "есть" т.е. неинформативный признак)

Серия дома (это косвенно определяет ряд другиз признаков, но само не влияет на цену, и серий домов очень большое множество)

Тип фундамента

Прописано (неинформативная колонка)

Вид из окон (т.к. дублирует колонку Вид из окна и даёт информацию по целому дому)

Санузел (дублирует информацию из колонки Санузел совмещён)

Количество лифтов (т.к. это зависит почти всегда только от признака Количество Этажей и вычисл. по формуле, т.е. признак лишний)

Несовершеннолетние собственники, Прописанные несовершеннолетние - юридические данные и почти не влияет на цену

Наименьшее кол-во этажей, холод. водоснаб., санузел отдельно, лоджия, детская площадка, европланировка, теплоснабжение, энергоснабжение, балкон, спортивная площадка, лоджии, водоотведение, Класс энергоэффективности, Количество собственников, Год ввода в эксплуатацию - мусорные данные которые могут ухудшить нашу модель ввиду неинформативности/отсутствия данных в 90%+ и более объектах

Количество балконов, мусоропровод - отсутствуют более чем в 75% объектах, решено удалить



In [203]:
cols_to_drop = ['Адрес','Подъезд','Описание',
'Балкон','Балконы','Лоджия','Лоджии',
'Всегоквартирв продаже','ФинансируетсяСберБанком','Материал стен','Количество этажей','Количество подъездов',
'Квартира','Лифт','Серия дома','Тип фундамента','Прописано','Несовершеннолетние собственники','Прописанные несовершеннолетние',
'Наименьшее количество этажей','Холодное водоснабжение','Санузел отдельно','Детская площадка','Европланировка','Теплоснабжение',
'Энергоснабжение','Спортивная площадка','Водоотведение','Класс энергоэффективности', 'Вид из окон', 'Санузел', 'Количество лифтов',
'Количество собственников', 'Горячее водоснабжение', 'Количество балконов', 'Мусоропровод', "Год ввода в эксплуатацию"]

df = df.drop(columns=cols_to_drop)

In [204]:
df.shape

(10481, 30)

In [205]:
df.to_csv('data_clean2.csv', index=False, encoding='utf-8-sig',sep=';')

Теперь займёмся обработкой каждого поля по-отдельности

Для начала заполним пропуски в данных

Для заполнения ячеек "Жилая площадь" и "Кухня" попробуем определить зависимость от площади квартиры (находим коэф. и заполняем пропуски путём перемножения этого коэфа на площадь квартиры)

In [206]:
cols = ['Площадь', 'Жилая', 'Кухня']

for col in cols:
    df[col] = pd.to_numeric(df[col].replace({'м²': '', 'м2': '', ',': '.'}, regex=True))

# коэффициент для жилой и кухни
coef_living = (df.loc[df['Жилая'].notna(), 'Жилая']/df.loc[df['Жилая'].notna(),'Площадь']).mean()
coef_kitchen = (df.loc[df['Кухня'].notna(), 'Кухня']/df.loc[df['Кухня'].notna(),'Площадь']).mean()

print(coef_living)

df.loc[df['Жилая'].isna(), 'Жилая'] = (df.loc[df['Жилая'].isna(),'Площадь'] * coef_living).round().astype(int)
df.loc[df['Кухня'].isna(), 'Кухня'] = (df.loc[df['Кухня'].isna(),'Площадь'] * coef_kitchen).round().astype(int)

df['Жилая'] = pd.to_numeric(df['Жилая'])
df['Кухня'] = pd.to_numeric(df['Кухня'])

0.5258359163106483


для признака "класс жилья" заполним пропуски модой (самым встреч. знач), а также используем label encoding для кодировки этого признака

In [207]:
classZhil = {'Стандарт':1,'Комфорт':2,'Бизнес':3,'Премиум':4}
df['Класс жилья'] = df['Класс жилья'].fillna(df['Класс жилья'].mode()[0])

df['Класс жилья'] = df['Класс жилья'].map(classZhil)

Признак "Ремонт" объединим с признаком "Отделка", создав новый признак "Состояние" (пропусков в признаках ремонт/отделка нет)


In [208]:
text = df['Ремонт'].astype(str) + ' ' + df['Отделка'].astype(str)

df['Состояние'] = 'Средний'
df.loc[text.str.contains('без'), 'Состояние'] = 'Без ремонта'
df.loc[text.str.contains('требует'), 'Состояние'] = 'Без ремонта'
df.loc[text.str.contains('чернов'), 'Состояние'] = 'Без ремонта'
df.loc[text.str.contains('предчист'), 'Состояние'] = 'Предчистовая'
df.loc[text.str.contains('космет'), 'Состояние'] = 'Косметический'
df.loc[text.str.contains('евро'), 'Состояние'] = 'Евроремонт'
df.loc[text.str.contains('дизайн'), 'Состояние'] = 'Евроремонт'
df.loc[text.str.contains('чист') & (text.str.contains('предчист') == False), 'Состояние'] = 'Чистовая'

In [209]:
remont_map = {'Без ремонта': 0, 'Предчистовая': 1, 'Средний': 2, 'Косметический': 3, 'Чистовая': 4, 'Евроремонт': 5}

df['Состояние'] = df['Состояние'].map(remont_map)

cols_droping = ['Ремонт','Отделка'] #удаляем теперь уже лишние колонки
df = df.drop(columns=cols_droping)

Строки с пропущенным признаком "Комнат" было решено удалить, т.к. он предоставляет большое значение для нас, и мы не можем заполнить пропуски с кол-вом комнат каким-либо значением, это может сильно испортить нашу модель (пропусков менее 10% от всех данных)

In [210]:
print(df['Комнат'].isna().sum())
df = df.dropna(subset=['Комнат'])
print(df['Комнат'].isna().sum())

df['Комнат'] = pd.to_numeric(df['Комнат'])

869
0


Заполним пропуски в колонке "Вид из окна" модой, а также закодируем с помощью OHE, т.к. для нас это важный признак и он может сильно влиять на цену (при этом это не ранговый столбец, т.к. в некоторых случаях вид на двор лучше чем вид на улицу, например, если на улице ж/д пути или трасса)

In [211]:
df['Вид из окна'] = df['Вид из окна'].fillna(df['Вид из окна'].mode()[0])
df = pd.get_dummies(df, columns=['Вид из окна'], dtype=int, drop_first=True)

Также обработаем ряд других, менее важных (по степени влияния на цену) признаков:

Кол-во санузлов больше всего зависит от площади квартиры и кол-во комнат, попробуем определить зависимость и заполнить пропуски с помощью линейной регрессии


In [212]:
#df['Санузел совмещён'] = df['Санузел совмещён'].fillna(df['Санузел совмещён'].mode()[0]) #заполняем кол-во туалетов самым встречающимся кол-во
#print(df.loc[df['Санузел совмещён'] == 3, 'Площадь'].quantile(0.25), df.loc[df['Санузел совмещён'] == 3, 'Площадь'].quantile(0.75))
#print(df.loc[df['Санузел совмещён'] == 2, 'Площадь'].quantile(0.25), df.loc[df['Санузел совмещён'] == 2, 'Площадь'].quantile(0.75))
#print(df.loc[df['Санузел совмещён'] == 1, 'Площадь'].quantile(0.25), df.loc[df['Санузел совмещён'] == 1, 'Площадь'].quantile(0.75))

train_priz_df = df[df['Санузел совмещён'].notna()]
prizX = train_priz_df[['Площадь', 'Комнат']]
prizY = train_priz_df['Санузел совмещён']
model = LinearRegression()
model.fit(prizX,prizY)
df.loc[df['Санузел совмещён'].isna(),'Санузел совмещён'] = model.predict(df.loc[df['Санузел совмещён'].isna(), ['Площадь','Комнат']]).round().astype(int)

In [213]:
print(model.coef_)

[-0.006317   0.3697989]


Поле аккредитация заполнено только у новостроек, поэтому добавим новый признак "ПервичноеЖилье" (может быть полезно при дальнейш. анализе признаков) и заменим на 0/1 признак аккредитации новостройки

Также обработаем пропуски и данные в признаках Продажаквартирчерез эскроу и Материалстен

In [214]:
print(df.loc[(df['Аккредитация'] == 'Аккредитован'), 'Аккредитация'].notna().sum())
print(df.loc[(df['Аккредитация'] == 'Не аккредитован'), 'Аккредитация'].notna().sum())


#добавили признак для первичек
df['ПервичноеЖилье'] = df['Аккредитация'].notna().astype(int)

#-1 значит что неприменимо к текущему объекту, и у этой строчки признак первичноежилье будет -1
df['Аккредитация'] = df['Аккредитация'].map({'Аккредитован' : 1, 'Не аккредитован' : 0}).fillna(-1)

#-1 значит что неприменимо к текущему объекту, и у этой строчки признак первичноежилье будет -1
df['Продажаквартирчерез эскроу'] = df['Продажаквартирчерез эскроу'].map({'Да' : 1, 'Нет' : 0}).fillna(-1)

#заполняем объекты предыдущим значением, т.к. чаще всего дома в одном районе
#затем делаем OHE, тк это категор. неранг. признак
df['Материалстен'] = df['Материалстен'].fillna(method='ffill')
df = pd.get_dummies(df, columns=['Материалстен'], dtype=int)

4104
288


/tmp/ipykernel_4322/248983998.py:16: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Материалстен'] = df['Материалстен'].fillna(method='ffill')


Теперь обработаем признак количество этажей и этаж

С кол-вом этажей проблем нет, а этаж нужно обработать, т.к. он в очень разных форматах, чаще всего как datatime и x/y и int, обработаем эти случаи

In [215]:
df['Количествоэтажей'] = df['Количествоэтажей'].fillna(df['Количествоэтажей'].mode()[0])#заполним пропуски в колво этажей модой
df['Количествоэтажей'] = pd.to_numeric(df['Количествоэтажей']).astype(int)

def clean_floor(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()

    #если признак вида x/y берем X
    if '/' in s:
        return int(s.split('/')[0].strip())

    #если это datatime, то значит берём month, тк это ошибка парсинга и месяц должен быть этажем
    dt = pd.to_datetime(s, dayfirst=True, errors='coerce')
    if pd.notna(dt):
        return dt.month

    # если просто число: "4", "19", "5"
    if s.isdigit():
        return int(x)
    #остальные случаи
    return np.nan


# применяем функцию и оставшиеся пропуски заполняем модой
df['Этаж'] = df['Этаж'].apply(clean_floor)
df['Этаж'] = df['Этаж'].fillna(df['Этаж'].mode()[0])

/tmp/ipykernel_4322/3160533855.py:14: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dt = pd.to_datetime(s, dayfirst=True, errors='coerce')


Удалим лишние символы в признаке "Время до метро"  и заполним пропуски (их менее 10%) медианой

UPD: нет, времяяя до метро не заполняем

In [216]:
#df['Время до метро'] = df['Время до метро'].str.replace(' мин.', '')
#df['Время до метро'] = pd.to_numeric(df['Время до метро'])
#df['Время до метро'] = df['Время до метро'].fillna(df['Время до метро'].median()) #заполним пропуски (их очень мало) медианой

Посмотрим и обработаем признаки тип сделки и тип жилья

In [217]:
#тип сделки закодируем с помощью OHE, т.к. это категориальный непорядковый признак
df = pd.get_dummies(df, columns=['Тип сделки'], dtype=int, drop_first=True)

#тип жилья также закодируем с помощью OHE, т.к. это категориальный непорядковый признак, но перед этим пропуски
#заполним значением "Квартира", т.к. жилые апартаменты с 2024г. практически в Москве перестали строить (и их в датасете менее 3% от
#общего кол-ва объектов), а пропущенные значения типа жилья только у новостроек
print((df['Тип жилья'] == 'Квартира').sum())
df['Тип жилья'] = df['Тип жилья'].fillna('Квартира')
df = pd.get_dummies(df, columns=['Тип жилья'], dtype=int, drop_first=True)


4924


Обработаем признаки: перепланировка, грузовой лифт и газ

In [218]:
#признак перепланировка закодируем так: если есть, то 0, если нет, то 1, также заполним пропуски значением "Нет" (значение Есть встречается в 0,5% объектах)
df['Перепланировка'] = df['Перепланировка'].fillna('Нет')
df['Перепланировка'] = df['Перепланировка'].map({'Есть' : 0, 'Нет' : 1})

#в категор. признаке "Грузовой лифт" заполним пропуски модой (знач. "Есть")
#и с помощью label encoding как 1/0 закодируем
df['Грузовой лифт'] = df['Грузовой лифт'].fillna('Есть')
df['Грузовой лифт'] = df['Грузовой лифт'].map({'Есть' : 1, 'Нет' : 0})

print((df['Газ'] == 'Есть').sum())
#в категор. признаке "Газ" заполним пропуски модой (значением "Нет")
#и с помощью label encoding как 1/0 закодируем этот признак
df['Газ'] = df['Газ'].fillna('Нет')
df['Газ'] = df['Газ'].map({'Есть' : 1, 'Нет' : 0})




1494


Обработаем признак "Лет в собственности", заполнив пропуски самым встреч. значением и закодировав его с помощью OHE (т.к. это не ранговый категор. признак)

In [219]:
df['Лет в собственности'] = df['Лет в собственности'].fillna(df['Лет в собственности'].mode()[0])
df = pd.get_dummies(df, columns=['Лет в собственности'], dtype=int, drop_first=True)

Для признака "Год постройки" заполним пропуски: если объект это первичной жильё, то заполн. медианой известных годов постройки по первичкам, а иначе медианой по вторичкам

В исходном датасете год постройки не указан ни для одного типа вторичного жилья, поэтому поставим в исходном признаке "-1" для тех объектов, где первичное жильё

In [220]:
maska = df['Год постройки'].isna()

med0 = df.loc[df['ПервичноеЖилье'] == 0, 'Год постройки'].median()

df.loc[(maska & (df['ПервичноеЖилье'] == 1)), 'Год постройки'] = -1
df.loc[(maska & (df['ПервичноеЖилье'] == 0)), 'Год постройки'] = med0

print(med1, med0)

nan 2004.0


Для признака "Количество квартир" поступим таким же образом, если объект это первичной жильё, то заполн. медианой кол-вом жилья по первичкам, а иначе по вторичкам установим знач. -1

In [221]:
med0 = df.loc[df['ПервичноеЖилье'] == 0, 'Количество квартир'].median()

maska = df['Количество квартир'].isna()

df.loc[(maska & (df['ПервичноеЖилье'] == 1)), 'Количество квартир'] = -1
df.loc[(maska & (df['ПервичноеЖилье'] == 0)), 'Количество квартир'] = med0

Для признака "Тип перекрытий" заполним пропуски модой (мода - "Железобетонное", встреч. практически во всех объектах с известным знач) и сделаем OHE (т.к. категориальный неранговый признак)

Для признака "высота потолков" приведём к числовому типу и заполним пропуски медианой

In [222]:
df['Тип перекрытий'] = df['Тип перекрытий'].fillna(df['Тип перекрытий'].mode()[0])
df = pd.get_dummies(df, columns=['Тип перекрытий'], dtype=int)


df['Высота потолков'] = pd.to_numeric(df['Высота потолков'].str.replace(',', '.',regex=False))
df['Высота потолков'] = df['Высота потолков'].fillna(df['Высота потолков'].median())

In [226]:
df.to_excel('data_clean6.xlsx', index=False)